# Uni-Mol 5-class odor classification — Colab-safe LoRA fine-tuning

In [ ]:
%pip install -q scikit-learn "unimol-tools==0.1.1" rdkit

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('molecular-foundation-model')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/FufanLu/molecular-foundation-model.git',
        str(repo_dir),
    ], check=True)
else:
    print(f'{repo_dir} already exists; reusing it.')

import sys
sys.path.insert(0, str(repo_dir.resolve()))

from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

df = encode_labels(clean_smiles(load_leffingwell()))
print(f'{len(df)} valid molecules')
label_distribution(df)

## Prepare Uni-Mol inputs once

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from unimol_tools.data import DataHub

torch.set_num_threads(2)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

smiles_list = df['smiles'].tolist()
print(f'Generating 3D inputs for {len(smiles_list)} molecules...')

# IMPORTANT: DataHub/ConformerGen defaults to multiprocessing on Linux.
# Forking after CUDA has been initialized is a common Colab deadlock.
hub = DataHub(
    data=smiles_list,
    task='repr',
    is_train=False,
    data_type='molecule',
    model_name='unimolv1',
    batch_size=4,
    remove_hs=False,
    use_cuda=torch.cuda.is_available(),
    use_ddp=False,
    use_gpu='0',
    multi_process=False,
)
features = hub.data['unimol_input']
Y = np.stack(df['y'].values).astype(np.float32)
indices = np.arange(len(features))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Prepared {len(features)} inputs; train={len(train_idx)}, test={len(test_idx)}')

## Build a differentiable Uni-Mol + LoRA model

In [ ]:
import torch.nn as nn
from unimol_tools.models import UniMolModel
from src.classifier.model import apply_lora

backbone = UniMolModel(
    output_dim=len(ALL_5_CLASSES),
    data_type='molecule',
    remove_hs=False,
).to(device)

# Freeze pretrained weights first. apply_lora adds trainable adapters.
for parameter in backbone.parameters():
    parameter.requires_grad = False

linear_leaf_names = {
    name.rsplit('.', 1)[-1]
    for name, module in backbone.named_modules()
    if isinstance(module, nn.Linear)
}
target_names = [
    name for name in ['in_proj', 'q_proj', 'k_proj', 'v_proj', 'query', 'key', 'value']
    if name in linear_leaf_names
]
if not target_names:
    raise RuntimeError(f'No q/k/v layers found. Linear layers: {sorted(linear_leaf_names)}')

n_replaced = apply_lora(backbone, r=4, alpha=8, target_names=target_names)
for parameter in backbone.classification_head.parameters():
    parameter.requires_grad = True
if n_replaced == 0:
    raise RuntimeError('LoRA matched zero layers.')

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f'LoRA layers replaced: {n_replaced}')
print(f'Trainable parameters: {trainable:,} / {total:,}')

## Train (no `get_repr` inside the loop)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

class MoleculeDataset(Dataset):
    def __init__(self, inputs, labels, indices):
        self.inputs = inputs
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = int(self.indices[position])
        return self.inputs[index], self.labels[index]

train_loader = DataLoader(
    MoleculeDataset(features, Y, train_idx),
    batch_size=4, shuffle=True, num_workers=0, pin_memory=False,
    collate_fn=backbone.batch_collate_fn,
)
test_loader = DataLoader(
    MoleculeDataset(features, Y, test_idx),
    batch_size=4, shuffle=False, num_workers=0, pin_memory=False,
    collate_fn=backbone.batch_collate_fn,
)

optimizer = torch.optim.AdamW(
    [p for p in backbone.parameters() if p.requires_grad],
    lr=2e-4, weight_decay=1e-4,
)
criterion = nn.BCEWithLogitsLoss()
epochs = 5

def evaluate(loader):
    backbone.eval()
    logits, labels = [], []
    with torch.no_grad():
        for batch_inputs, batch_labels in loader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            logits.append(backbone(**batch_inputs).cpu())
            labels.append(batch_labels.float())
    probs = torch.sigmoid(torch.cat(logits)).numpy()
    labels = torch.cat(labels).numpy()
    return f1_score(labels, (probs > 0.5).astype(int), average='macro', zero_division=0)

for epoch in range(epochs):
    backbone.train()
    total_loss = 0.0
    for batch_inputs, batch_labels in train_loader:
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(backbone(**batch_inputs), batch_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(backbone.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch {epoch + 1:02d}  loss {total_loss / len(train_loader):.4f}  f1_macro {evaluate(test_loader):.4f}')

In [ ]:
backbone.eval()
all_logits = []
with torch.no_grad():
    for batch_inputs, _ in test_loader:
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        all_logits.append(backbone(**batch_inputs).cpu())
preds_bin = (torch.sigmoid(torch.cat(all_logits)).numpy() > 0.5).astype(int)
Y_test = Y[test_idx]
print(f"{'Class':<12} {'F1':>7}")
print('-' * 21)
for i, cls in enumerate(ALL_5_CLASSES):
    print(f'{cls:<12} {f1_score(Y_test[:, i], preds_bin[:, i], zero_division=0):>7.4f}')
print('-' * 21)
print(f"{'macro avg':<12} {f1_score(Y_test, preds_bin, average='macro', zero_division=0):>7.4f}")

In [ ]:
torch.save({
    'model_state_dict': backbone.state_dict(),
    'target_names': target_names,
    'classes': ALL_5_CLASSES,
}, 'odor_lora.pt')
print('Saved odor_lora.pt')
from google.colab import files
files.download('odor_lora.pt')